[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_11/NB_bishop_ch11_graphical_models.ipynb)

# Chapter 11 -- Graphical Models

This notebook explores **probabilistic graphical models** following Bishop Chapter 11:
- Bayesian networks (directed graphical models)
- D-separation and conditional independence
- Naive Bayes classifier
- Hidden Markov Models (HMMs)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from collections import defaultdict

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

np.random.seed(42)
print(f'NumPy version: {np.__version__}')

## 1. Bayesian Network Fundamentals

A Bayesian network is a directed acyclic graph (DAG) where nodes represent random variables and edges encode conditional dependencies.

In [ ]:
class BayesianNetwork:
    """Simple Bayesian network with discrete variables."""
    def __init__(self):
        self.nodes = {}
        self.parents = defaultdict(list)
        self.children = defaultdict(list)
        self.cpts = {}

    def add_node(self, name, values):
        self.nodes[name] = values

    def add_edge(self, parent, child):
        self.parents[child].append(parent)
        self.children[parent].append(child)

    def set_cpt(self, node, cpt):
        self.cpts[node] = cpt

    def sample(self, n=1000):
        """Forward sampling from the network."""
        order = self._topological_sort()
        samples = []
        for _ in range(n):
            s = {}
            for node in order:
                if not self.parents[node]:
                    probs = list(self.cpts[node].values())
                    vals = list(self.cpts[node].keys())
                else:
                    parent_vals = tuple(s[p] for p in self.parents[node])
                    cpt_entry = self.cpts[node][parent_vals]
                    probs = list(cpt_entry.values())
                    vals = list(cpt_entry.keys())
                s[node] = np.random.choice(vals, p=probs)
            samples.append(s)
        return samples

    def _topological_sort(self):
        visited = set()
        order = []
        def dfs(node):
            if node in visited:
                return
            visited.add(node)
            for p in self.parents[node]:
                dfs(p)
            order.append(node)
        for n in self.nodes:
            dfs(n)
        return order

print('BayesianNetwork class defined.')

## 2. Classic Alarm Network

Burglary and Earthquake can trigger an Alarm, which may cause Mary or John to call.

In [ ]:
bn = BayesianNetwork()
for name in ['Burglary', 'Earthquake', 'Alarm', 'JohnCalls', 'MaryCalls']:
    bn.add_node(name, [0, 1])

bn.add_edge('Burglary', 'Alarm')
bn.add_edge('Earthquake', 'Alarm')
bn.add_edge('Alarm', 'JohnCalls')
bn.add_edge('Alarm', 'MaryCalls')

bn.set_cpt('Burglary', {1: 0.001, 0: 0.999})
bn.set_cpt('Earthquake', {1: 0.002, 0: 0.998})
bn.set_cpt('Alarm', {
    (1, 1): {1: 0.95, 0: 0.05},
    (1, 0): {1: 0.94, 0: 0.06},
    (0, 1): {1: 0.29, 0: 0.71},
    (0, 0): {1: 0.001, 0: 0.999}
})
bn.set_cpt('JohnCalls', {
    (1,): {1: 0.90, 0: 0.10},
    (0,): {1: 0.05, 0: 0.95}
})
bn.set_cpt('MaryCalls', {
    (1,): {1: 0.70, 0: 0.30},
    (0,): {1: 0.01, 0: 0.99}
})

samples = bn.sample(50000)
alarm_count = sum(1 for s in samples if s['Alarm'] == 1)
john_given_alarm = sum(1 for s in samples if s['JohnCalls'] == 1 and s['Alarm'] == 1) / max(alarm_count, 1)
print(f'P(Alarm=1) approx: {alarm_count/50000:.4f}')
print(f'P(JohnCalls=1|Alarm=1) approx: {john_given_alarm:.4f}')

In [ ]:
# Visualize the alarm network structure
fig, ax = plt.subplots(figsize=(8, 6))
positions = {
    'Burglary': (1, 3), 'Earthquake': (3, 3),
    'Alarm': (2, 2),
    'JohnCalls': (1, 1), 'MaryCalls': (3, 1)
}
for name, (x, y) in positions.items():
    ax.add_patch(plt.Circle((x, y), 0.35, fill=True, alpha=0.3, color='#1a3a6e'))
    ax.text(x, y, name, ha='center', va='center', fontsize=9, fontweight='bold')

edges = [('Burglary', 'Alarm'), ('Earthquake', 'Alarm'),
         ('Alarm', 'JohnCalls'), ('Alarm', 'MaryCalls')]
for p, c in edges:
    px, py = positions[p]
    cx, cy = positions[c]
    ax.annotate('', xy=(cx, cy + 0.35), xytext=(px, py - 0.35),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#cd0000'))

ax.set_xlim(0, 4); ax.set_ylim(0, 4)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Alarm Bayesian Network', fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'bishop_ch11_bayesian_net')
plt.show()

## 3. D-Separation

D-separation determines conditional independence in a Bayesian network. Three canonical structures:
1. **Chain**: A -> B -> C (A _|_ C | B)
2. **Fork**: A <- B -> C (A _|_ C | B)
3. **Collider**: A -> B <- C (A _|_ C, but NOT when B is observed)

In [ ]:
def demonstrate_dsep(structure, n=100000):
    """Demonstrate d-separation with sampling."""
    A = np.random.choice([0, 1], size=n, p=[0.5, 0.5])

    if structure == 'chain':
        B = np.array([np.random.choice([0,1], p=[0.3,0.7]) if a==1 else np.random.choice([0,1], p=[0.8,0.2]) for a in A])
        C = np.array([np.random.choice([0,1], p=[0.2,0.8]) if b==1 else np.random.choice([0,1], p=[0.9,0.1]) for b in B])
    elif structure == 'fork':
        B_root = np.random.choice([0,1], size=n, p=[0.5,0.5])
        A = np.array([np.random.choice([0,1], p=[0.3,0.7]) if b==1 else np.random.choice([0,1], p=[0.8,0.2]) for b in B_root])
        B = B_root
        C = np.array([np.random.choice([0,1], p=[0.2,0.8]) if b==1 else np.random.choice([0,1], p=[0.9,0.1]) for b in B])
    else:
        C = np.random.choice([0, 1], size=n, p=[0.5, 0.5])
        B = np.array([1 if (a==1 or c==1) else np.random.choice([0,1], p=[0.9,0.1]) for a, c in zip(A, C)])

    corr_marginal = np.corrcoef(A.astype(float), C.astype(float))[0, 1]
    mask = B == 1
    corr_cond = np.corrcoef(A[mask].astype(float), C[mask].astype(float))[0, 1] if mask.sum() > 10 else float('nan')
    return corr_marginal, corr_cond

for s in ['chain', 'fork', 'collider']:
    m, c = demonstrate_dsep(s)
    print(f'{s:10s} | Corr(A,C) marginal: {m:+.4f} | Corr(A,C|B=1): {c:+.4f}')

In [ ]:
# Visualize the three canonical structures
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
titles = ['Chain: A -> B -> C', 'Fork: A <- B -> C', 'Collider: A -> B <- C']
edge_lists = [
    [('A', 'B'), ('B', 'C')],
    [('B', 'A'), ('B', 'C')],
    [('A', 'B'), ('C', 'B')]
]
pos_struct = {'A': (0, 0), 'B': (1, 0), 'C': (2, 0)}

for ax, title, edges in zip(axes, titles, edge_lists):
    for name, (x, y) in pos_struct.items():
        ax.add_patch(plt.Circle((x, y), 0.2, fill=True, alpha=0.3, color='#1a3a6e'))
        ax.text(x, y, name, ha='center', va='center', fontsize=14, fontweight='bold')
    for p, c in edges:
        px, py = pos_struct[p]; cx, cy = pos_struct[c]
        ax.annotate('', xy=(cx - 0.22*np.sign(cx-px), cy),
                    xytext=(px + 0.22*np.sign(cx-px), py),
                    arrowprops=dict(arrowstyle='->', lw=2, color='gray'))
    ax.set_xlim(-0.5, 2.5); ax.set_ylim(-0.8, 0.8)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(title, fontsize=11, fontweight='bold')

plt.suptitle('D-Separation: Three Canonical Structures', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'bishop_ch11_dseparation')
plt.show()

## 4. Joint Probability by Enumeration

In [ ]:
# Rain-Sprinkler-WetGrass network
P_C = {True: 0.5, False: 0.5}
P_S_given_C = {(True, True): 0.1, (True, False): 0.9, (False, True): 0.5, (False, False): 0.5}
P_R_given_C = {(True, True): 0.8, (True, False): 0.2, (False, True): 0.2, (False, False): 0.8}
P_W_given_SR = {
    (True, True, True): 0.99, (True, True, False): 0.01,
    (True, False, True): 0.90, (True, False, False): 0.10,
    (False, True, True): 0.90, (False, True, False): 0.10,
    (False, False, True): 0.00, (False, False, False): 1.00
}

def joint_prob(c, s, r, w):
    return P_C[c] * P_S_given_C[(c, s)] * P_R_given_C[(c, r)] * P_W_given_SR[(s, r, w)]

total = sum(joint_prob(c, s, r, w)
            for c in [True, False] for s in [True, False]
            for r in [True, False] for w in [True, False])
print(f'Sum of all joint probabilities: {total:.6f}')

P_W_true = sum(joint_prob(c, s, r, True)
               for c in [True, False] for s in [True, False] for r in [True, False])
print(f'P(Wet Grass = True) = {P_W_true:.4f}')

P_R_true_W_true = sum(joint_prob(c, s, True, True)
                      for c in [True, False] for s in [True, False])
print(f'P(Rain=True | Wet Grass=True) = {P_R_true_W_true / P_W_true:.4f}')

## 5. Naive Bayes Classifier

In [ ]:
class GaussianNaiveBayes:
    """Gaussian Naive Bayes classifier from scratch."""
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.params = {}
        self.priors = {}
        for c in self.classes:
            X_c = X[y == c]
            self.params[c] = {'mean': X_c.mean(axis=0), 'var': X_c.var(axis=0) + 1e-9}
            self.priors[c] = len(X_c) / len(X)
        return self

    def _log_likelihood(self, X):
        log_liks = np.zeros((X.shape[0], len(self.classes)))
        for idx, c in enumerate(self.classes):
            mean, var = self.params[c]['mean'], self.params[c]['var']
            log_liks[:, idx] = -0.5 * np.sum(np.log(2*np.pi*var) + (X - mean)**2 / var, axis=1)
        return log_liks

    def predict(self, X):
        log_posterior = self._log_likelihood(X) + np.log([self.priors[c] for c in self.classes])
        return self.classes[np.argmax(log_posterior, axis=1)]

    def score(self, X, y):
        return np.mean(self.predict(X) == y)

print('GaussianNaiveBayes class defined.')

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)

gnb = GaussianNaiveBayes().fit(X_train, y_train)
print(f'Train accuracy: {gnb.score(X_train, y_train):.4f}')
print(f'Test accuracy:  {gnb.score(X_test, y_test):.4f}')

In [ ]:
# Decision boundary for Naive Bayes (first 2 features)
from matplotlib.colors import ListedColormap

X_2d = X_iris[:, :2]
gnb_2d = GaussianNaiveBayes().fit(X_2d, y_iris)

x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
Z = gnb_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

colors_bn = ['#1a3a6e', '#cd0000', '#228B22']
cmap_bg = ListedColormap(['#1a3a6e22', '#cd000022', '#228B2222'])

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_bg, levels=[-0.5, 0.5, 1.5, 2.5])
for idx, name in enumerate(iris.target_names):
    mask = y_iris == idx
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=colors_bn[idx], label=name,
               edgecolors='white', s=50, alpha=0.8)
ax.set_xlabel(iris.feature_names[0])
ax.set_ylabel(iris.feature_names[1])
ax.set_title('Naive Bayes Decision Boundaries (Iris)', fontweight='bold')
ax.legend()
plt.tight_layout()
save_fig(fig, 'bishop_ch11_naive_bayes')
plt.show()

## 6. Comparison with scikit-learn

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

sklearn_gnb = GaussianNB().fit(X_train, y_train)
y_pred_sk = sklearn_gnb.predict(X_test)
y_pred_ours = gnb.predict(X_test)

print(f'Scikit-learn accuracy: {(y_pred_sk == y_test).mean():.4f}')
print(f'Our accuracy:          {(y_pred_ours == y_test).mean():.4f}')
print(f'Predictions match: {np.all(y_pred_ours == y_pred_sk)}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_ours, target_names=iris.target_names))

## 7. Hidden Markov Model -- Forward Algorithm

Hidden states: Sunny, Rainy. Observations: Walk, Shop, Clean.

In [ ]:
class HMM:
    """Simple Hidden Markov Model with forward and Viterbi algorithms."""
    def __init__(self, states, observations, start_prob, trans_prob, emit_prob):
        self.states = states
        self.observations = observations
        self.pi = start_prob
        self.A = trans_prob
        self.B = emit_prob
        self.n_states = len(states)

    def forward(self, obs_seq):
        T = len(obs_seq)
        alpha = np.zeros((T, self.n_states))
        obs_idx = self.observations.index(obs_seq[0])
        alpha[0] = self.pi * self.B[:, obs_idx]
        for t in range(1, T):
            obs_idx = self.observations.index(obs_seq[t])
            for j in range(self.n_states):
                alpha[t, j] = np.sum(alpha[t-1] * self.A[:, j]) * self.B[j, obs_idx]
        return alpha, np.sum(alpha[-1])

    def viterbi(self, obs_seq):
        T = len(obs_seq)
        delta = np.zeros((T, self.n_states))
        psi = np.zeros((T, self.n_states), dtype=int)
        obs_idx = self.observations.index(obs_seq[0])
        delta[0] = np.log(self.pi + 1e-300) + np.log(self.B[:, obs_idx] + 1e-300)
        for t in range(1, T):
            obs_idx = self.observations.index(obs_seq[t])
            for j in range(self.n_states):
                trans = delta[t-1] + np.log(self.A[:, j] + 1e-300)
                psi[t, j] = np.argmax(trans)
                delta[t, j] = trans[psi[t, j]] + np.log(self.B[j, obs_idx] + 1e-300)
        path = np.zeros(T, dtype=int)
        path[-1] = np.argmax(delta[-1])
        for t in range(T-2, -1, -1):
            path[t] = psi[t+1, path[t+1]]
        return [self.states[i] for i in path]

states = ['Sunny', 'Rainy']
observations = ['Walk', 'Shop', 'Clean']
start_prob = np.array([0.6, 0.4])
trans_prob = np.array([[0.7, 0.3], [0.4, 0.6]])
emit_prob = np.array([[0.6, 0.3, 0.1], [0.1, 0.4, 0.5]])

hmm = HMM(states, observations, start_prob, trans_prob, emit_prob)
print(f'States: {states}, Observations: {observations}')

In [ ]:
test_sequences = [
    ['Walk', 'Walk', 'Walk'],
    ['Clean', 'Clean', 'Clean'],
    ['Walk', 'Shop', 'Clean'],
    ['Walk', 'Walk', 'Clean', 'Shop', 'Walk'],
]

for seq in test_sequences:
    alpha, prob = hmm.forward(seq)
    viterbi_path = hmm.viterbi(seq)
    print(f'Obs: {seq}')
    print(f'  P(seq|model) = {prob:.6f}')
    print(f'  Viterbi path: {viterbi_path}\n')

In [ ]:
# Visualize forward probabilities
seq = ['Walk', 'Shop', 'Clean', 'Walk', 'Shop']
alpha, prob = hmm.forward(seq)
alpha_norm = alpha / alpha.sum(axis=1, keepdims=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(len(seq)), alpha[:, 0], 'o-', color='#DAA520', label='Sunny', linewidth=2)
ax1.plot(range(len(seq)), alpha[:, 1], 's-', color='#1a3a6e', label='Rainy', linewidth=2)
ax1.set_xticks(range(len(seq)))
ax1.set_xticklabels(seq)
ax1.set_xlabel('Observation')
ax1.set_ylabel('Forward probability')
ax1.set_title(f'Forward Probabilities (P = {prob:.4f})', fontweight='bold')
ax1.legend()

x_pos = np.arange(len(seq))
ax2.bar(x_pos - 0.15, alpha_norm[:, 0], 0.3, color='#DAA520', label='Sunny')
ax2.bar(x_pos + 0.15, alpha_norm[:, 1], 0.3, color='#1a3a6e', label='Rainy')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(seq)
ax2.set_xlabel('Observation')
ax2.set_ylabel('Posterior P(state | obs)')
ax2.set_title('Filtered State Posteriors', fontweight='bold')
ax2.legend()

fig.tight_layout()
save_fig(fig, 'bishop_ch11_hmm_forward')
plt.show()

## 8. HMM Regime Detection (Gaussian Emissions)

In [ ]:
# Simulate bull/bear market regimes
np.random.seed(123)
T = 200
A_regime = np.array([[0.95, 0.05], [0.10, 0.90]])
means_regime = [0.05, -0.03]
stds_regime = [0.01, 0.02]

true_states = np.zeros(T, dtype=int)
obs_regime = np.zeros(T)
true_states[0] = np.random.choice(2, p=[0.6, 0.4])
obs_regime[0] = np.random.normal(means_regime[true_states[0]], stds_regime[true_states[0]])

for t in range(1, T):
    true_states[t] = np.random.choice(2, p=A_regime[true_states[t-1]])
    obs_regime[t] = np.random.normal(means_regime[true_states[t]], stds_regime[true_states[t]])

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(obs_regime, color='#1a3a6e', lw=0.8)
axes[0].set_ylabel('Returns')
axes[0].set_title('Simulated Market Returns with Hidden Regimes', fontweight='bold')

axes[1].fill_between(range(T), 0, 1, where=true_states==0, alpha=0.3, color='#228B22', label='Bull')
axes[1].fill_between(range(T), 0, 1, where=true_states==1, alpha=0.3, color='#cd0000', label='Bear')
axes[1].set_ylabel('Regime')
axes[1].set_xlabel('Time')
axes[1].legend()

plt.tight_layout()
save_fig(fig, 'bishop_ch11_hmm_regimes')
plt.show()

## 9. Markov Blanket Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
nodes_mb = {
    'X': (2, 2), 'P1': (1, 3), 'P2': (3, 3),
    'C1': (1, 1), 'C2': (3, 1),
    'CP1': (0, 1), 'CP2': (4, 1), 'O': (2, 0)
}
blanket = {'P1', 'P2', 'C1', 'C2', 'CP1', 'CP2'}

for name, (x, y) in nodes_mb.items():
    color = '#DAA520' if name == 'X' else ('#cd0000' if name in blanket else 'lightgray')
    ax.add_patch(plt.Circle((x, y), 0.3, fill=True, alpha=0.5, color=color, ec='black', lw=1.5))
    ax.text(x, y, name, ha='center', va='center', fontsize=10, fontweight='bold')

dag_edges = [('P1','X'),('P2','X'),('X','C1'),('X','C2'),('CP1','C1'),('CP2','C2'),('C1','O')]
for p, c in dag_edges:
    px, py = nodes_mb[p]; cx, cy = nodes_mb[c]
    dx, dy = cx-px, cy-py
    d = np.sqrt(dx**2+dy**2)
    ax.annotate('', xy=(cx-0.3*dx/d, cy-0.3*dy/d), xytext=(px+0.3*dx/d, py+0.3*dy/d),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))

ax.set_xlim(-1, 5); ax.set_ylim(-0.8, 4)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Markov Blanket of X (red = blanket, gold = target)', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'bishop_ch11_markov_blanket')
plt.show()

## 10. Spam Detection with Naive Bayes

In [ ]:
np.random.seed(42)
vocab = ['free', 'money', 'offer', 'click', 'meeting', 'report', 'data', 'project', 'urgent', 'winner']
n_vocab = len(vocab)
spam_probs = np.array([0.2, 0.15, 0.15, 0.12, 0.02, 0.02, 0.02, 0.02, 0.15, 0.15])
ham_probs  = np.array([0.02, 0.02, 0.02, 0.02, 0.18, 0.18, 0.18, 0.18, 0.05, 0.05])
spam_probs /= spam_probs.sum()
ham_probs /= ham_probs.sum()

n_docs = 300
labels_sp = np.random.choice([0, 1], size=n_docs, p=[0.6, 0.4])
X_text = np.zeros((n_docs, n_vocab))
for i in range(n_docs):
    p = spam_probs if labels_sp[i] == 1 else ham_probs
    words = np.random.choice(n_vocab, size=20, p=p)
    for w in words:
        X_text[i, w] += 1

X_tr, X_te, y_tr, y_te = train_test_split(X_text, labels_sp, test_size=0.3, random_state=42)
nb_text = GaussianNaiveBayes().fit(X_tr, y_tr)
print(f'Spam classifier accuracy: {nb_text.score(X_te, y_te):.2%}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(n_vocab)
w = 0.35
ax.bar(x - w/2, spam_probs, w, label='Spam', color='#cd0000', alpha=0.7)
ax.bar(x + w/2, ham_probs, w, label='Ham', color='#1a3a6e', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(vocab, rotation=45, ha='right')
ax.set_ylabel('Word Probability')
ax.set_title('Naive Bayes Spam Filter: Word Distributions', fontweight='bold')
ax.legend()
plt.tight_layout()
save_fig(fig, 'bishop_ch11_spam_words')
plt.show()

## Summary

- **Bayesian networks** encode joint distributions via directed acyclic graphs
- **D-separation** provides a graphical test for conditional independence
- **Naive Bayes** is a simple but powerful classifier based on conditional independence
- **HMMs** model temporal sequences with hidden states (forward + Viterbi algorithms)
- The **Markov blanket** defines the minimal set of nodes that shields a variable from the rest

In [ ]:
takeaways = [
    '1. Bayesian networks encode joint distributions compactly via conditional independence.',
    '2. d-separation provides a graphical criterion for reading off conditional independencies.',
    '3. Naive Bayes assumes feature independence given the class, yet often works well.',
    '4. The forward algorithm computes P(observations) in O(T * N^2) time.',
    '5. Graphical models unify directed (Bayesian) and undirected (Markov) representations.'
]
print('Key Takeaways:')
for t in takeaways:
    print(t)